# Analysis-ready opportunity dataset

This notebook prepares one row per frozen runner-control opportunity, audits feature cutoffs and action assignments, and records the frozen exploratory plan inputs. It reuses protocol-v1.1 actions/windows and cached public feeds. It does not fit a runner-threat, pitcher-response, timing-response, or causal model. The video gate remains `validation_pending`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.prepare_opportunity_dataset import run_preparation
pd.set_option('display.max_colwidth', 100)
ROOT

WindowsPath('C:/Users/isaac/CodingProjects/runner-threat-pitcher-tempo/runner-threat-pitcher-tempo')

## Rebuild reviewed artifacts

The preparation reads only existing manifests, ignored cached public feeds, and the executed frozen audit outputs. Validation fails closed before files are written if any frozen accounting, cutoff, assignment, split, or raw-disengagement check fails.

In [2]:
outputs = run_preparation(ROOT, write=True)
opportunities = outputs['opportunities']
print(f"Prepared {len(opportunities):,} rows and {opportunities.shape[1]} columns.")

Prepared 12,228 rows and 140 columns.


## Population and response accounting

These are inspected development data, not a representative sample or final test. Pickoff and steal labels remain separate and may coexist.

In [3]:
outputs['population_summary']

,metric,value,definition
0,manifest_rows,212,source rows before game deduplication
1,unique_cached_games,211,deduplicated game population
2,feasibility_audit_games,30,schedule-stratified audit source
3,tempo_benchmark_games,182,games containing appearances by six selected benchmark pitchers
4,overlap_games,1,deduplicated overlap
5,first_game_date,2024-03-28,cached development coverage
6,last_game_date,2024-09-29,cached development coverage
7,unique_dates,130,cached development coverage
8,unique_teams,30,game participation
9,unique_venues,30,cached development coverage


In [4]:
outputs['exclusion_summary']

,category,windows,treatment,reason
0,frozen_action_population,12228,retained,all first-base-only windows
1,terminal_without_next_pitch,315,retained,later pitch is not an eligibility condition
2,runner_identity_changed_within_window,234,flagged_for_model_review,occupancy window retained; opening runner no longer describes full exposure
3,unknown_pickoff_initiator,1,preserved_separately,not coded as pitcher or catcher
4,unknown_action_classification,0,preserved_separately,not silently coerced
5,known_state_conflict,0,preserved_separately,game 746690 PA 38 contributes no windows
6,opening_catcher_not_reconstructed,12228,missing_predictor,requires verified chronological defense reconstruction
7,complete_pre_game_runner_history_unavailable,12228,blocks_threat_model,selective cache cannot provide denominators
8,preliminary_structural_response_ineligible,235,not_dropped_from_table,identity/classification flags only
9,secondary_action_free_take_interval,4584,secondary_subset,conditioning on no action can select on post-threat behavior


## Correctness and leakage checks

All response events must follow the feature cutoff. Terminal windows are retained. A feed action record may be assigned to at most one window. Pre-game histories stay explicitly missing because the selective cache cannot supply valid denominators.

In [5]:
outputs['validation_checks']

,check,passed,detail
0,unique_opportunity_id,True,stable one-row key
1,window_count,True,12228
2,pickoff_window_identity,True,675
3,steal_window_identity,True,205
4,combined_window_identity,True,2
5,union_window_identity,True,878
6,action_assignments_match,True,no duplicate or missing action joins
7,action_assignment_unique_across_windows,True,1008 assigned records
8,terminal_windows_retained,True,315
9,cutoff_event_order,True,all response orders exceed cutoff


In [6]:
disengagement = outputs['disengagement_checks']
pd.DataFrame({
    'measure': ['action records checked', 'raw ordinals available', 'raw ordinal matches', 'continuation records'],
    'value': [
        len(disengagement),
        int(disengagement['raw_number_available'].sum()),
        int(disengagement['raw_number_matches'].fillna(False).sum()),
        int(disengagement['continuation_of_prior_pickoff_record'].sum()),
    ],
})

,measure,value
0,action records checked,1155
1,raw ordinals available,1126
2,raw ordinal matches,1126
3,continuation records,2


In [7]:
high_risk_fields = [
    'feature_cutoff_event_order',
    'recorded_disengagements_since_reset_at_open',
    'runner_attempt_rate_shrunk',
    'recorded_pitcher_pickoff_indicator',
    'recorded_catcher_pickoff_indicator',
    'recorded_steal_attempt_indicator',
    'recorded_no_pickoff_or_steal_indicator',
    'secondary_event_start_interval_seconds',
]
dictionary = outputs['feature_dictionary']
assert len(dictionary) == opportunities.shape[1]
dictionary.set_index('field').loc[high_risk_fields].reset_index()

,field,group,definition,source,availability_time,analysis_role,denominator,missing_data_handling,leakage_rule
0,feature_cutoff_event_order,A_opening_information,Last event order whose information may enter predictors; all response events have a strictly lar...,chronological live-feed reconstruction,at_feature_cutoff,candidate_predictor_or_context,all opportunity windows,use explicit missing indicator; do not infer catcher from final lineup,may be used only if known at or before feature cutoff
1,recorded_disengagements_since_reset_at_open,A_opening_information,Recorded pitcher pickoffs plus step-offs since the most recent runner advance in the plate appea...,chronological live-feed reconstruction,at_feature_cutoff,candidate_predictor_or_context,all opportunity windows,use explicit missing indicator; do not infer catcher from final lineup,may be used only if known at or before feature cutoff
2,runner_attempt_rate_shrunk,A_pre_game_history,Training-estimated shrinkage of prior recorded steal attempts divided by prior eligible opportun...,future complete historical opportunity dataset,strictly_before_game,planned_threat_predictor,runner-games with complete prior coverage,currently missing by design; pooled prior for new/sparse players after collection,never derive from current/future game or final-season totals
3,recorded_pitcher_pickoff_indicator,B_window_actions_outcomes,One or more feed-recorded pickoff throws initiated by the pitcher after cutoff and before closur...,linked action records within window,after_window_closes,outcome_or_diagnostic,all opportunity windows unless field states another denominator,retain zero as no recorded event; unknown initiator remains separate,never use as a predictor for the same window
4,recorded_catcher_pickoff_indicator,B_window_actions_outcomes,One or more feed-recorded catcher-initiated pickoff throws; separate from primary response.,linked action records within window,after_window_closes,outcome_or_diagnostic,all opportunity windows unless field states another denominator,retain zero as no recorded event; unknown initiator remains separate,never use as a predictor for the same window
5,recorded_steal_attempt_indicator,B_window_actions_outcomes,One or more feed-recorded stolen-base or caught-stealing attempts within the response window.,linked action records within window,after_window_closes,outcome_or_diagnostic,all opportunity windows unless field states another denominator,retain zero as no recorded event; unknown initiator remains separate,never use as a predictor for the same window
6,recorded_no_pickoff_or_steal_indicator,B_window_actions_outcomes,No pickoff or steal is recorded in the feed window; this is not confirmed real-world absence.,linked action records within window,after_window_closes,outcome_or_diagnostic,all opportunity windows unless field states another denominator,retain zero as no recorded event; unknown initiator remains separate,never use as a predictor for the same window
7,secondary_event_start_interval_seconds,B_window_closure,Following pitch event start minus opening pitch event start for action-free same-PA windows foll...,chronological window reconstruction,after_window_closes,outcome_or_eligibility,all windows; timing value only on labeled secondary subset,retain terminal/missing next pitch; timing otherwise missing,post-cutoff; action-free eligibility is selection on post-threat behavior


## Proposed expansion and locked chronology

No broader download occurs here. The current cache has no eligible final-test rows. Dates must be rechecked against official schedule responses at collection time, and every returned regular-season game in the bounds must be included.

In [8]:
outputs['proposed_splits']

,split,start_date,end_date,fit_parameters,tuning_allowed,use,game_separation,chronological_order
0,history_only,2022-04-07,2022-10-05,False,False,pregame priors only; old rule environment retained as labeled history,whole_game,strict
1,training,2023-03-30,2023-10-01,True,True,"fit baselines, regularized models, preprocessing, and shrinkage",whole_game,strict
2,validation,2024-03-20,2024-09-30,False,True,select features and penalties; cached subset is already inspected,whole_game,strict
3,final_test,2025-03-18,2025-09-28,False,False,locked one-time evaluation; whole games only,whole_game,strict


In [9]:
outputs['additional_data_needed']

,priority,data,scope,reason,status
0,1,Complete regular-season schedules and live feeds,2022 history plus every 2023-2025 regular-season game,"Build unbiased denominators and pre-game histories; preserve URLs, retrieval times, and checksums",required_before_modeling
1,2,Chronological catcher identity,Starting defensive alignments plus every catcher substitution for collected games,Opening catcher cannot be inferred from the final lineup and is a planned context variable,required_before_catcher_adjustment
2,3,Forward runner opportunity/attempt/success histories,All eligible windows dated strictly before each game,Generate leakage-safe attempt and conditional-success predictors with training-estimated shrinkage,required_before_modeling
3,4,As-of player metadata,Handedness and roster/team identity available on each game date,Opening context and sparse/new-player handling,required_before_modeling
4,5,Lagged Statcast sprint-speed snapshots,Prior-season value or documented pre-game snapshot only,Optional threat feature; final-season values would leak future information,optional_after_primary_baseline
5,6,Expanded independent action review,Event-complete source or video sample for positive and no-recorded-action windows,Estimate label sensitivity where feasible; absence is currently unvalidated,recommended_for_interpretation
6,7,Continuous video release annotations,Frozen video sample only,Required only for any release-tempo claim; not required for recorded-pickoff analysis,required_only_for_timing_secondary


## Interpretation

The table is ready for schema review and collection planning, not for modeling. All 12,228 opportunities lack complete pre-game runner histories, and the 211-game union is selectively sampled. After complete 2022-2025 regular-season collection and forward history construction, an explicitly labeled recorded-pitcher-pickoff model may proceed if completeness checks pass. The action model does not require release-video validation. Timing remains deferred; the 4,584 action-free take intervals are event-start sensitivities only, and conditioning on no action changes the population. No Zac Gallen interval identities are guessed, and the video gate remains `validation_pending`.